# De la exploración al reporte de mediciones

La primera notebook nos permitió comprobar formas, seleccionar rondas y resumir
columnas. Ahora usaremos esas operaciones como funciones reutilizables desde la
notebook y desde `main.py`, sin reconstruir la lógica en cada celda.

## Contenido

1. Cargar y validar la forma.
2. Seleccionar rondas completas.
3. Resumir por sala e interpretar el resultado.
4. Añadir una selección propia.

Abre esta notebook en VS Code desde `02-reporte-mediciones` y selecciona el kernel de `.venv`.


In [2]:
import json
from pathlib import Path

import numpy as np

from measurements.processing import (
    ROOM_NAMES,
    load_readings,
    select_complete_rows,
    summarize,
)

project_dir = Path.cwd()
data_dir = project_dir.parent / "datos"
if not (project_dir / "main.py").is_file() or not data_dir.is_dir():
    raise FileNotFoundError(
        "Start the notebook from the 02-reporte-mediciones project directory"
    )

## 1. Cargar y validar la forma

`load_readings` comprueba el encabezado y lee una matriz con tres columnas.
La comprobación impide asignar un resumen a otra sala si cambia el orden del CSV.
Abre el módulo en el editor e identifica qué parte lee y cuál valida.

[Lectura con loadtxt](https://numpy.org/doc/stable/reference/generated/numpy.loadtxt.html).

In [3]:
readings = load_readings(data_dir / "readings.csv")
print(ROOM_NAMES)
print(readings.shape)
print(readings[:2])
assert readings.shape == (8, 3)

('north_room', 'south_room', 'storage_room')
(8, 3)
[[22. 24. 20.]
 [23. 25. 21.]]


## 2. Seleccionar rondas completas

La función devuelve dos resultados: las filas seleccionadas y la cantidad de
rechazos. Esto permite conservar evidencia del filtrado. Ni el arreglo original
ni el CSV se modifican.

In [4]:
quality_readings = load_readings(data_dir / "readings_quality.csv")
original = quality_readings.copy()
complete, rejected = select_complete_rows(quality_readings)
print(complete)
print(f"Rejected rows: {rejected}")
assert rejected == 2
assert not np.shares_memory(complete, quality_readings)
np.testing.assert_array_equal(quality_readings, original)

[[22. 24. 20.]
 [24. 26. 22.]]
Rejected rows: 2


### Comprobación: ninguna ronda completa

No calcularemos medias sobre una matriz vacía. La función comunica el problema
con una excepción; la notebook puede capturarla para discutir el caso y continuar.

In [5]:
invalid_readings = np.array([[np.nan, 24, 20], [22, np.inf, 21]])
try:
    select_complete_rows(invalid_readings)
except ValueError as error:
    print(f"Expected error: {error}")
else:
    raise AssertionError("Expected an error for data without complete rows")

Expected error: No complete readings remain


## 3. Resumir por sala

Las operaciones numéricas se aplican por eje. El pequeño ciclo de la función
`summarize` asigna nombres a los tres resultados y convierte los escalares a
`float` de Python para serializarlos en JSON. No recorre las mediciones para
calcular sus medias.

In [6]:
report = summarize(complete)
print(json.dumps(report, indent=2, allow_nan=False))
np.testing.assert_allclose(
    [report[name]["mean_c"] for name in ROOM_NAMES], [23, 25, 21]
)

{
  "north_room": {
    "mean_c": 23.0,
    "min_c": 22.0,
    "max_c": 24.0
  },
  "south_room": {
    "mean_c": 25.0,
    "min_c": 24.0,
    "max_c": 26.0
  },
  "storage_room": {
    "mean_c": 21.0,
    "min_c": 20.0,
    "max_c": 22.0
  }
}


### Ejercicio 10: explicar el cambio de medias

Compara el reporte de las ocho rondas originales con el de las dos rondas
completas del archivo con errores. Describe qué datos entraron en cada cálculo.
No son dos momentos de una instalación real: son archivos de práctica distintos.

In [7]:
# Ejercicio 10: explicar el cambio de medias
# Compare the summaries and write your interpretation.

report_full = summarize(load_readings(data_dir / "readings.csv"))  # 8 rondas
report_quality = summarize(complete)  # 2 rondas completas

print("Completo (8 rondas):", {k: v["mean_c"] for k, v in report_full.items()})
print("Con errores (2 rondas):", {k: v["mean_c"] for k, v in report_quality.items()})

# Explicacion

"""
El reporte completo usa las 8 rondas de readings.csv y da medias de 25.5, 27.5 y 23.5 °C. El de readings_quality.csv usa solo las 2 rondas con 
las tres salas finitas (se rechazan 2 por valores no finitos) y da 23, 25 y 21 °C. La diferencia no refleja un cambio en una misma instalación: 
son archivos de práctica distintos y el segundo promedia menos rondas y diferentes, por eso bajan las medias.
"""

Completo (8 rondas): {'north_room': 25.5, 'south_room': 27.5, 'storage_room': 23.5}
Con errores (2 rondas): {'north_room': 23.0, 'south_room': 25.0, 'storage_room': 21.0}


'\nEl reporte completo usa las 8 rondas de readings.csv y da medias de 25.5, 27.5 y 23.5 °C. El de readings_quality.csv usa solo las 2 rondas con \nlas tres salas finitas (se rechazan 2 por valores no finitos) y da 23, 25 y 21 °C. La diferencia no refleja un cambio en una misma instalación: \nson archivos de práctica distintos y el segundo promedia menos rondas y diferentes, por eso bajan las medias.\n'

## 4. Añadir una selección propia

### Ejercicio 11: seleccionar rondas cálidas

Resuelve la [práctica](../PRACTICA.md) creando `select_warm_rounds` en un módulo
nuevo. Reinicia el kernel después de guardar el módulo y comprueba aquí las filas
seleccionadas antes de ejecutar sus pruebas.

Para umbral `27`, deben quedar tres rondas del archivo original. Para umbral
`100`, la selección es vacía y no se debe calcular su media.

In [8]:
# Import your selection function and verify the results here.

# Ejercicio 11: seleccionar rondas cálidas
from measurements.selection import select_warm_rounds

full = load_readings(data_dir / "readings.csv")
complete_full, _ = select_complete_rows(
    full
)  # "después de seleccionar filas completas"

warm = select_warm_rounds(complete_full, 27)
print(warm)
np.testing.assert_array_equal(warm, [[27, 29, 25], [28, 30, 26], [29, 31, 27]])

empty = select_warm_rounds(complete_full, 100)
print("Forma con umbral 100:", empty.shape)
assert empty.shape == (0, 3)

print("Media selección (umbral 27):", warm.mean(axis=0))
print("Media conjunto completo:", full.mean(axis=0))

# Explicacion

"""
Con umbral 27 entran solo las rondas con sala norte ≥ 27 (las tres últimas: 27, 28, 29), así que la media de la selección 
se calcula sobre esas 3 filas y da [28, 30, 26]. La media del conjunto completo usa las 8 rondas y da [25.5, 27.5, 23.5]. 
Con umbral 100 ninguna ronda cumple: la selección queda (0, 3) y no se calcula su media, porque promediar un arreglo vacío 
daría nan (no hay filas sobre las cuales promediar).

"""

[[27. 29. 25.]
 [28. 30. 26.]
 [29. 31. 27.]]
Forma con umbral 100: (0, 3)
Media selección (umbral 27): [28. 30. 26.]
Media conjunto completo: [25.5 27.5 23.5]


'\nCon umbral 27 entran solo las rondas con sala norte ≥ 27 (las tres últimas: 27, 28, 29), así que la media de la selección \nse calcula sobre esas 3 filas y da [28, 30, 26]. La media del conjunto completo usa las 8 rondas y da [25.5, 27.5, 23.5]. \nCon umbral 100 ninguna ronda cumple: la selección queda (0, 3) y no se calcula su media, porque promediar un arreglo vacío \ndaría nan (no hay filas sobre las cuales promediar).\n\n'

## Verificación final

Desde la terminal del proyecto ejecuta `uv run --locked python main.py` para
generar los archivos, y `uv run --locked python -m pytest` para comprobar las
funciones. Después reinicia el kernel y ejecuta toda la notebook.
